# Graph Questions

Reusable answers for graph-mining questions. Change the parameters in the cells marked **Parameters** to answer similar questions later.


## Imports

In [30]:
import numpy as np
from scipy import sparse
from sknetwork.data import load_netset
from sknetwork.path import get_distances, get_shortest_path
from sknetwork.utils import get_neighbors
from sknetwork.ranking import PageRank
from sknetwork.embedding import Spectral


## Load Data

In [31]:
cinema = load_netset('cinema')

# In the cinema graph, rows are movies and columns are actors.
biadjacency = cinema.biadjacency
movies = cinema.names_row
actors = cinema.names_col


Parsing files...
Done.


In [32]:
openflights = load_netset('openflights')

openflights_adjacency = openflights.adjacency
openflights_names = openflights.names


Parsing files...
Done.


In [33]:
wikivitals = load_netset('wikivitals')

wikivitals_adjacency = wikivitals.adjacency
wikivitals_names = wikivitals.names
wikivitals_labels = wikivitals.labels
wikivitals_names_labels = wikivitals.names_labels


Parsing files...
Done.


## Helper Functions

In [34]:
def find_exact_name(names, query):
    """Return the index of one exact name, or show close matches if not found."""
    matches = np.where(names == query)[0]
    if len(matches) == 1:
        return int(matches[0])
    if len(matches) > 1:
        raise ValueError(f'Multiple exact matches for {query}: {names[matches]}')

    # Helpful when you want to change a parameter and misspell a name.
    close = np.where(np.char.find(np.char.lower(names.astype(str)), query.lower()) >= 0)[0]
    raise ValueError(f'No exact match for {query}. Close matches: {names[close[:10]]}')


def bacon_number(actor_name, reference_actor='Kevin Bacon'):
    """Compute an actor's Bacon number without building the actor-actor graph."""
    actor_id = find_exact_name(actors, actor_name)
    reference_id = find_exact_name(actors, reference_actor)

    # Distances are measured on movie-actor paths: actor -> movie -> actor is 2 hops.
    _, actor_distances = get_distances(biadjacency, source_col=reference_id)
    distance = actor_distances[actor_id]

    if distance < 0:
        return None
    return distance // 2


In [35]:
def personalized_airport_ranking(airport_weights):
    """Rank airports by Personalized PageRank from weighted source airports."""
    source_ids = {
        find_exact_name(openflights_names, airport): weight
        for airport, weight in airport_weights.items()
    }

    scores = PageRank().fit_predict(openflights_adjacency, weights=source_ids)
    ranking = np.argsort(scores)[::-1]
    return ranking, scores, source_ids


In [ ]:
def count_incoming_links(article_names):
    """Count how many Wikivitals articles link to each target article."""
    counts = {}
    both=0
    for article in article_names:
        article_id = find_exact_name(wikivitals_names, article)
        counts[article] = int(wikivitals_adjacency[:, article_id].sum())
        a2 = find_exact_name(wikivitals_names, "Senegal")
        for node in wikivitals_adjacency[:, article_id]:
            if wikivitals_adjacency[node, a2] > 0:
                both+=1
        print(both)
    return counts, both


In [47]:
both=0
article_id = find_exact_name(wikivitals_names, "France")
# counts["France"] = int(wikivitals_adjacency[:, article_id].sum())
a2 = find_exact_name(wikivitals_names, "Senegal")
for node in range(wikivitals_adjacency[:, article_id].shape[0]):
    if wikivitals_adjacency[node, a2] > 0:
        both+=1
print(both)


302


In [37]:
def rank_topical_categories(n_components=20, matrix='transition'):
    """Rank Wikivitals categories by average pairwise cosine similarity.

    matrix can be 'transition' for row-normalized links, or 'adjacency' for raw links.
    """
    if matrix == 'transition':
        # Row-normalize A so each row gives transition probabilities.
        row_sums = np.array(wikivitals_adjacency.sum(axis=1)).flatten()
        inv_row_sums = np.zeros_like(row_sums, dtype=float)
        inv_row_sums[row_sums > 0] = 1 / row_sums[row_sums > 0]
        input_matrix = sparse.diags(inv_row_sums).dot(wikivitals_adjacency)
    elif matrix == 'adjacency':
        input_matrix = wikivitals_adjacency
    else:
        raise ValueError("matrix must be 'transition' or 'adjacency'")

    embedding = Spectral(n_components, normalized=True).fit_transform(input_matrix)

    topicality = []
    for label_id, category in enumerate(wikivitals_names_labels):
        indices = np.where(wikivitals_labels == label_id)[0]
        category_embedding = embedding[indices]

        # Since vectors are normalized, average cosine similarity is ||centroid||^2.
        centroid = category_embedding.mean(axis=0)
        score = float(centroid @ centroid)
        topicality.append((category, score, len(indices)))

    return sorted(topicality, key=lambda item: item[1], reverse=True)


## Question 1: Bacon Number

What is the Bacon number of Marion Cotillard?


### Parameters

In [20]:
x =  8 + 2 +2+2+10+2+10
print(x)

36


In [48]:
ACTOR_NAME = 'Rossy de Palma'
REFERENCE_ACTOR = 'Kevin Bacon'


### Answer

In [49]:
number = bacon_number(ACTOR_NAME, REFERENCE_ACTOR)
print(f'The Bacon number of {ACTOR_NAME} is {number}.')


The Bacon number of Rossy de Palma is 2.


## Question 2: Personalized PageRank Airports

In the Openflights graph, what is the closest airport from Charles de Gaulle International Airport and Beijing Capital International Airport in terms of Personalized PageRank?

Take respective weights 2 and 1 for these two airports.


### Parameters

In [38]:
AIRPORT_WEIGHTS = {
    'Vancouver International Airport': 1,
    'Calgary International Airport': 1,
}


### Answer

In [39]:
ranking, scores, source_ids = personalized_airport_ranking(AIRPORT_WEIGHTS)

# The source airports rank highest by construction, so we skip them.
closest_airport = next(airport for airport in ranking if airport not in source_ids)

print(openflights_names[closest_airport])


Kamloops Airport


## Question 3: Incoming Links in Wikivitals

How many articles of Wikivitals have links to each of the following articles: France, Japan and Egypt?


### Parameters

In [45]:
TARGET_ARTICLES = ['France']


### Answer

In [ ]:
incoming_counts = count_incoming_links(TARGET_ARTICLES)

for article, count in incoming_counts.items():
    print(f'{article}: {count}')


IndexError: bool index 0 has shape (10011, 10011) instead of (1, 1)

## Question 4: Most Topical Wikivitals Category

Consider the spectral embedding of Wikivitals in dimension 20, based on the transition matrix. We say that a category is topical if its average pairwise cosine similarity is high, close to 1. What is the most topical category?


### Parameters

In [15]:
EMBEDDING_DIMENSION = 20
EMBEDDING_MATRIX = 'transition'  # change to 'adjacency' to use raw links


### Answer

In [16]:
topicality = rank_topical_categories(EMBEDDING_DIMENSION, EMBEDDING_MATRIX)

best_category, best_score, n_articles = topicality[0]
print(f'Matrix used: {EMBEDDING_MATRIX}')
print(f'Most topical category: {best_category}')
print(f'Average cosine similarity: {best_score:.3f}')
print(f'Number of articles: {n_articles}')


Matrix used: transition
Most topical category: Mathematics
Average cosine similarity: 0.644
Number of articles: 300


## Question 5: Personalized PageRank From One Airport

In the Openflights graph, what is the best ranked airport in terms of Personalized PageRank starting from Vancouver International Airport, after this airport?

Type the name of the airport.


### Parameters

In [42]:
SOURCE_AIRPORT = 'Vancouver International Airport'


### Answer

In [43]:
ranking, scores, source_ids = personalized_airport_ranking({SOURCE_AIRPORT: 1})

# The source airport is ranked first by construction, so we return the next one.
best_after_source = next(airport for airport in ranking if airport not in source_ids)

print(openflights_names[best_after_source])
print(openflights_names[ranking[:15]])

Calgary International Airport
['Vancouver International Airport' 'Calgary International Airport'
 'Kamloops Airport' 'Phoenix Sky Harbor International Airport'
 'San Francisco International Airport' 'Denver International Airport'
 'Dallas Fort Worth International Airport'
 'Salt Lake City International Airport'
 "Chicago O'Hare International Airport"
 'Licenciado Gustavo Díaz Ordaz International Airport'
 'Los Angeles International Airport'
 'George Bush Intercontinental Houston Airport'
 'Lester B. Pearson International Airport'
 'McCarran International Airport' 'Prince George Airport']
